### Phase 2 prediction

based on logits/final-layer embeddings?

In [ ]:
import json
import torch
import re
import os
import math
import random
from tqdm import tqdm
import numpy as np
from scipy import sparse
import matplotlib.pyplot as plt
from torch.nn import functional as F
from transformers import AutoTokenizer, AutoModelForCausalLM, utils
from datasets import load_dataset

Dataset preparation


For attention_weight things:

- Dataset structure:

    ```json
    {
        "system_params": {
            "temperature": float,
            "top_k": int,
            "repetition_penalty": float,
            "max_seq_len": int
        },
        "model_arch": {
            "num_layers": int,
            "num_heads": int
        },
        "samples": [
            {
                "layer": int,
                "head": int,
                "attention_matrix": np.array(seq_len, seq_len),
                "seq_pos": int,
            },
        ],
        "label": {
            "remaining_tokens": int,
            "over_max_seq_len": bool
        }
    }
    ```
- Task: predict the remaining tokens (and over_max_seq_len) for each sample in the dataset.
  - Regression task
  - Models can be used: MLP, Transformer, RNN, LSTM, GRU, etc.

Here we use **MLP** and train it with the dataset. (maybe incorrect)

In [2]:
from torch.utils.data import Dataset

def commpress_attention_matrix(attention_matrix):
    """
    half precision
    sparse matrix
    Compress the attention matrix by removing the diagonal and upper triangular part
    Not yet: SVG compression
    """
    seq_len = attention_matrix.shape[0]
    mask = np.tri(seq_len, dtype=bool)
    attention_matrix = attention_matrix * mask
    
    attention_matrix = attention_matrix.astype(np.float16)
    sparse_matrix = sparse.coo_matrix(attention_matrix)
    return sparse_matrix

def encode_params(params):
    """
    Encode system parameters into a numpy array / 32-bit integer
    """
    assert 0.1 <= params['temperature'] <= 0.9
    assert 1 <= params['top_k'] <= 255
    assert 1.0 <= params['repetition_penalty'] <= 1.6
    assert 100 <= params['max_seq_len'] <= 25500
    
    # encode
    temp_enc = int(np.interp(params['temperature'], [0.1, 0.9], [0, 255]))
    topk_enc = params['top_k']
    rep_enc = int(np.interp(params['repetition_penalty'], [1.0, 1.6], [0, 255]))
    maxlen_enc = params['max_seq_len'] // 100
    
    return np.uint32(
        (temp_enc << 24) | 
        (topk_enc << 16) | 
        (rep_enc << 8) | 
        maxlen_enc
    )

def decode_params(encoded):
    """
    Decode system parameters from a numpy array / 32-bit integer
    """
    return {
        'temperature': ((encoded >> 24) & 0xFF) / 255 * 0.8 + 0.1,
        'top_k': (encoded >> 16) & 0xFF,
        'repetition_penalty': ((encoded >> 8) & 0xFF) / 255 * 0.6 + 1.0,
        'max_seq_len': (encoded & 0xFF) * 100
    }

class CompressedDataset(Dataset):
    def __init__(self, feature_dir):
        self.samples = []
        for fname in os.listdir(feature_dir):
            data = np.load(os.path.join(feature_dir, fname), allow_pickle=True)
            for feature, label in zip(data['features'], data['labels']):
                self.samples.append((feature, label))
                
    def __len__(self):
        return len(self.samples)
    
    def __getitem__(self, idx):
        feature, label = self.samples[idx]
        
        # 解码参数
        params = decode_params(feature['encoded_params'])
        
        # 重建注意力矩阵
        attn_matrix = sparse.coo_matrix(
            (feature['attn_data'], 
            (feature['attn_row'], feature['attn_col'])),
            shape=feature['attn_shape']
        ).toarray().astype(np.float32)
        
        # 构建训练样本
        return {
            'attn_matrix': torch.tensor(attn_matrix),
            'layer': torch.tensor(feature['layer']),
            'head': torch.tensor(feature['head']),
            'seq_pos': torch.tensor(feature['seq_pos'] / params['max_seq_len']),
            'temperature': torch.tensor(params['temperature']),
            'top_k': torch.tensor(params['top_k']),
            'rep_penalty': torch.tensor(params['repetition_penalty']),
            'label': torch.tensor(label['remaining_tokens'])
        }